# Lab 52 — QNLP: Quantum Natural Language Processing

El **QNLP** aplica la teoría de categorías (DisCoCat) para representar frases como circuitos cuánticos. La gramática de pregrupos mapea la estructura sintáctica de una oración en un diagrama de tensores, que luego se compila a un circuito IQP parametrizado.

En este laboratorio implementamos desde cero los conceptos fundamentales sin requerir el paquete `lambeq`:
1. **Pregrupos y diagramas** — representación categorial de frases
2. **Circuito IQP** — implementación manual con Qiskit
3. **Entrenamiento** — clasificación binaria positivo/negativo
4. **Evaluación** — comparativa con clasificador clásico (Naive Bayes)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from scipy.optimize import minimize
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score
np.random.seed(42)

print('Qiskit:', __import__('qiskit').__version__)

## 1. Pregrupos y Tipos Gramaticales

En la gramática de pregrupos de Lambek, cada palabra tiene un **tipo** que describe sus requisitos de composición:
- `n` = nombre/sustantivo
- `s` = frase completa
- `n·r` = necesita un `n` a la derecha para reducirse
- `n·l` = necesita un `n` a la izquierda

Para **SVO** (sujeto–verbo–objeto): `[n, n·r·s·l·n, n]` → `s`

In [ ]:
# Tipos de pregrupo: cada palabra tiene dimensión = número de qubits en el circuito IQP
# noun: 1 qubit, transitive_verb: 2 qubits (conecta sujeto y objeto)

def iqp_noun(params, label='noun'):
    """Circuito IQP para un nombre (1 qubit, 1 parámetro)."""
    qc = QuantumCircuit(1, name=label)
    qc.h(0)
    qc.rz(params[0], 0)
    qc.h(0)
    return qc

def iqp_verb(params, label='verb'):
    """Circuito IQP para un verbo transitivo (2 qubits, 3 parámetros)."""
    qc = QuantumCircuit(2, name=label)
    qc.h([0, 1])
    qc.rz(params[0], 0)
    qc.rz(params[1], 1)
    qc.cx(0, 1)
    qc.rz(params[2], 1)
    qc.cx(0, 1)
    qc.h([0, 1])
    return qc

def sentence_circuit(noun1_params, verb_params, noun2_params):
    """Circuito SVO completo (4 qubits): sujeto [0], verbo [1,2], objeto [3]."""
    qc = QuantumCircuit(4)
    # Sujeto (qubit 0)
    n1 = iqp_noun(noun1_params)
    qc.compose(n1, qubits=[0], inplace=True)
    # Verbo (qubits 1,2)
    v = iqp_verb(verb_params)
    qc.compose(v, qubits=[1, 2], inplace=True)
    # Objeto (qubit 3)
    n2 = iqp_noun(noun2_params)
    qc.compose(n2, qubits=[3], inplace=True)
    # Contracción: post-selección en |0⟩ para qubits 1,2 (traza sobre wire cups)
    qc.cx(0, 1)  # cup sujeto-verbo
    qc.cx(3, 2)  # cup verbo-objeto
    return qc

# Ejemplo: circuito para "Alice ama Bob"
alice_params = np.array([0.5])       # Alice
loves_params = np.array([0.3, 0.8, 0.4])  # ama
bob_params   = np.array([0.7])       # Bob

qc_ex = sentence_circuit(alice_params, loves_params, bob_params)
print("Circuito SVO (4 qubits):")
print(qc_ex.draw(output='text', fold=-1))

## 2. Dataset de Frases y Predicción

Usamos un dataset toy de frases positivas (amor, amistad) y negativas (odio, traición). Cada frase tiene estructura SVO con vocabulario fijo.

In [ ]:
# Dataset: (sujeto_idx, verbo_idx, objeto_idx, etiqueta)
# Verbos: 0=ama(+), 1=ayuda(+), 2=odia(-), 3=engaña(-)
# Sujetos/objetos: Alice(0), Bob(1), Carlos(2), Diana(3)

DATASET = [
    (0, 0, 1, 1), (0, 1, 2, 1), (1, 0, 3, 1), (2, 1, 0, 1),
    (3, 0, 0, 1), (1, 1, 1, 1), (2, 0, 3, 1), (3, 1, 2, 1),
    (0, 2, 1, 0), (0, 3, 2, 0), (1, 2, 3, 0), (2, 3, 0, 0),
    (3, 2, 0, 0), (1, 3, 1, 0), (2, 2, 3, 0), (3, 3, 2, 0),
]
print(f"Dataset: {len(DATASET)} frases, 8 positivas y 8 negativas")

def get_sentence_prob(params_all, subj_idx, verb_idx, obj_idx):
    """
    Calcula P(etiqueta=1) = |⟨00|ψ⟩|² (probabilidad de post-selección).
    params_all: [noun_params × 4, verb_params × 4] concatenados
    """
    n_noun_params = 1
    n_verb_params = 3
    noun_params = params_all[:4 * n_noun_params].reshape(4, n_noun_params)
    verb_params = params_all[4:].reshape(4, n_verb_params)
    
    qc = sentence_circuit(
        noun_params[subj_idx],
        verb_params[verb_idx],
        noun_params[obj_idx]
    )
    sv = Statevector(qc)
    # P(s) = suma de |amps|² con qubits internos (1,2) en |00⟩ tras traza parcial
    probs = sv.probabilities()
    # Mapeo: qubit 0 = sujeto, qubit 3 = objeto → resultado en (q0, q3)
    # Post-selección qubits 1,2 = 0 → índices 0b0x0y = 0,1,4,5 (contando q3210)
    p_pos = probs[0] + probs[1]   # |00 00⟩ + |00 01⟩
    p_total = sum(probs[i] for i in [0,1,4,5])  # post-selección q1=0, q2=0
    return p_pos / (p_total + 1e-12)

# Inicializar parámetros aleatoriamente
n_params = 4*1 + 4*3  # 4 noun × 1 param + 4 verb × 3 params = 16
theta_init = np.random.uniform(0, 2*np.pi, n_params)
print(f"Número de parámetros: {n_params}")

## 3. Entrenamiento con Optimización

Minimizamos la entropía cruzada binaria sobre el dataset de entrenamiento usando COBYLA.

In [ ]:
def binary_cross_entropy(params, dataset):
    """BCE loss sobre el dataset."""
    total_loss = 0.0
    for subj, verb, obj, label in dataset:
        p = np.clip(get_sentence_prob(params, subj, verb, obj), 1e-7, 1-1e-7)
        total_loss -= label * np.log(p) + (1-label) * np.log(1-p)
    return total_loss / len(dataset)

train_data = DATASET[:12]
test_data  = DATASET[12:]

loss_history = []
def callback(xk):
    loss_history.append(binary_cross_entropy(xk, train_data))

print("Entrenando circuito QNLP...")
result = minimize(
    binary_cross_entropy, theta_init, args=(train_data,),
    method='COBYLA',
    options={'maxiter': 200, 'rhobeg': 0.3},
    callback=callback
)
theta_opt = result.x
print(f"Loss final: {result.fun:.4f}")

# Evaluación
def predict(params, dataset, threshold=0.5):
    preds = []
    for subj, verb, obj, _ in dataset:
        p = get_sentence_prob(params, subj, verb, obj)
        preds.append(1 if p > threshold else 0)
    return preds

y_train_pred = predict(theta_opt, train_data)
y_test_pred  = predict(theta_opt, test_data)
y_train_true = [d[3] for d in train_data]
y_test_true  = [d[3] for d in test_data]
print(f"\nAccuracy train: {accuracy_score(y_train_true, y_train_pred):.2%}")
print(f"Accuracy test:  {accuracy_score(y_test_true, y_test_pred):.2%}")

In [ ]:
# Comparativa con clasificador clásico Naive Bayes
# Feature vector: (subj_idx, verb_idx, obj_idx) → one-hot o numérico
X_nb_train = np.array([[s,v,o] for s,v,o,_ in train_data])
y_nb_train = np.array([d[3] for d in train_data])
X_nb_test  = np.array([[s,v,o] for s,v,o,_ in test_data])
y_nb_test  = np.array([d[3] for d in test_data])

gnb = GaussianNB(); gnb.fit(X_nb_train, y_nb_train)
acc_nb_train = gnb.score(X_nb_train, y_nb_train)
acc_nb_test  = gnb.score(X_nb_test, y_nb_test)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(loss_history, color='steelblue', lw=2)
axes[0].set_xlabel('Iteración COBYLA'); axes[0].set_ylabel('BCE Loss')
axes[0].set_title('Convergencia del entrenamiento QNLP')
axes[0].grid(alpha=0.3)

methods = ['QNLP\n(train)', 'QNLP\n(test)', 'Naive Bayes\n(train)', 'Naive Bayes\n(test)']
accs = [
    accuracy_score(y_train_true, y_train_pred),
    accuracy_score(y_test_true, y_test_pred),
    acc_nb_train, acc_nb_test
]
colors_bar = ['steelblue', 'steelblue', 'darkorange', 'darkorange']
alphas = [1.0, 0.6, 1.0, 0.6]
bars2 = axes[1].bar(methods, accs, color=colors_bar, alpha=1.0)
for bar, a in zip(bars2, alphas): bar.set_alpha(a)
axes[1].set_ylim(0, 1.1); axes[1].set_ylabel('Accuracy')
axes[1].set_title('QNLP vs Naive Bayes — Dataset Toy')
axes[1].axhline(0.5, color='gray', ls='--', lw=1, label='Azar')
for b, v in zip(bars2, accs):
    axes[1].text(b.get_x()+b.get_width()/2, v+0.02, f'{v:.0%}', ha='center', fontsize=10)
plt.tight_layout(); plt.show()

print("\n¿Ventaja cuántica en QNLP?")
print("En datasets toy pequeños, los circuitos IQP pueden igualar o superar modelos clásicos.")
print("La ventaja real requeriría frases más largas y hardware cuántico de alta fidelidad.")